# Traffic-YOLO — VisDrone Fine-Tuning (Colab)

Bu notebook `models/best.pt`'yi VisDrone train split'i üzerinde daha fazla epoch ile fine-tune eder ve checkpoint'leri Google Drive'a yazar.

**Varsayımlar (önemli):** Orijinal eğitim notebook'u kayıp; `train_config.yaml`'daki `batch=16` ve `imgsz=640` değerleri gerçek orijinal koşuyla birebir aynı olmayabilir, makul varsayılanlardır. Farklı olduğunu biliyorsan `train_config.yaml`'ı güncelle ya da `--batch`/`--imgsz` ile override et.

**Sıra:** Drive mount → repo clone/pull → veri indir+dönüştür → **duman testi (5 epoch)** → (elle onay) **asıl koşu (150 epoch)**.

## 1. Google Drive'ı mount et
Checkpoint'ler ve eğitim çıktıları `/content/drive/MyDrive/Traffic-YOLO-runs/` altına yazılacak (Colab çalışma zamanı sıfırlansa bile kaybolmaz).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RUNS_DIR = '/content/drive/MyDrive/Traffic-YOLO-runs'
import os
os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)
print('Checkpoint klasörü:', DRIVE_RUNS_DIR)

## 2. Repoyu clone/pull et ve bağımlılıkları kur

**Not:** Bu hücrenin çalışması için `src/train.py`, `src/visdrone_convert.py`, `visdrone_full.yaml`, `train_config.yaml` içeren commit'lerin GitHub'a **push edilmiş** olması gerekir.

In [ ]:
REPO_URL = 'https://github.com/loopBreakerr/Traffic-YOLO.git'
REPO_DIR = '/content/Traffic-YOLO'

import os
if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 3. GPU kontrolü

In [ ]:
!nvidia-smi

## 4. VisDrone train + val split'lerini indir

Resmi Ultralytics asset URL'lerinden indirilir (val, `src/evaluate.py` için zaten aynı mantıkla hazırlanmıştı; train de aynı düzende). `model.train()` her epoch sonunda val split'i üzerinde de doğrulama yaptığı için val de burada gerekli.

In [ ]:
import os, urllib.request

os.makedirs('data/VisDrone', exist_ok=True)

ASSET_BASE = 'https://github.com/ultralytics/assets/releases/download/v0.0.0'
downloads = {
    'train': f'{ASSET_BASE}/VisDrone2019-DET-train.zip',
    'val': f'{ASSET_BASE}/VisDrone2019-DET-val.zip',
}

for split, url in downloads.items():
    dest = f'data/VisDrone/VisDrone2019-DET-{split}.zip'
    if os.path.exists(dest):
        print(f'{dest} zaten var, indirme atlandı.')
        continue
    print(f'İndiriliyor: {url}')
    urllib.request.urlretrieve(url, dest)
    size_mb = os.path.getsize(dest) / 1024 / 1024
    print(f'{dest}: {size_mb:.2f} MB')

In [ ]:
import zipfile

for split in ('train', 'val'):
    zip_path = f'data/VisDrone/VisDrone2019-DET-{split}.zip'
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('data/VisDrone')
    print(f'Açıldı: {zip_path}')

## 5. YOLO formatına dönüştür

`src/visdrone_convert.py` — alfabetik class remap dahil (bkz. `MODEL_CLASS_NAMES`). `--cleanup` ham `VisDrone2019-DET-{split}/` klasörünü siler, sadece `images/`+`labels/` kalır.

In [ ]:
!python -m src.visdrone_convert --root data/VisDrone --split train --cleanup
!python -m src.visdrone_convert --root data/VisDrone --split val --cleanup

In [ ]:
import os
for split in ('train', 'val'):
    img_dir = f'data/VisDrone/images/{split}'
    lbl_dir = f'data/VisDrone/labels/{split}'
    print(split, 'images:', len(os.listdir(img_dir)), 'labels:', len(os.listdir(lbl_dir)))

## 6. Sınıf sırası doğrulaması (opsiyonel ama önerilir)
`src/train.py` bunu zaten otomatik yapıyor; burada erken görmek için manuel de çalıştırılabilir.

In [ ]:
from src.visdrone_convert import verify_class_order
verify_class_order('models/best.pt')
print('Sınıf sırası OK.')

## 7. DUMAN TESTİ — 5 epoch

Bu hücre otomatik çalıştırılabilir (kısa sürer). Amaç: script'in hatasız bitip Drive'a checkpoint yazdığını doğrulamak. **Asıl 150 epoch'luk koşuyu BAŞLATMAZ.**

In [ ]:
!python -m src.train \
  --epochs 5 --patience 20 \
  --project {DRIVE_RUNS_DIR} --name smoke_test_5ep

In [ ]:
# Duman testinin Drive'a gerçekten checkpoint yazdığını doğrula
import os
ckpt_dir = f'{DRIVE_RUNS_DIR}/smoke_test_5ep/weights'
print(os.listdir(ckpt_dir))
assert os.path.exists(f'{ckpt_dir}/last.pt'), 'last.pt bulunamadı - duman testi checkpoint yazmamış olabilir!'
print('OK: last.pt Drive\'a yazıldı.')

---
## ⚠️ DURDUR — Asıl koşuyu otomatik çalıştırma

Yukarıdaki duman testi **hatasız bitti** ve `last.pt` Drive'da göründüyse devam et.

Aşağıdaki hücre **150 epoch**'luk asıl eğitimi başlatır ve saatler sürebilir. "Run All" ile otomatik tetiklenmemesi için bilinçli olarak ayrı bırakıldı — **bu hücreyi elle, tek başına çalıştır.**

In [ ]:
# BİLEREK burada: elle çalıştırılmadan önce asıl koşu başlamasın.
RUN_FULL_TRAINING = False  # elle True yap, sonra hücreyi çalıştır

if not RUN_FULL_TRAINING:
    raise SystemExit('RUN_FULL_TRAINING = True yapmadan bu hücreyi çalıştırma.')

!python -m src.train \
  --epochs 150 --patience 20 \
  --project {DRIVE_RUNS_DIR} --name exp1_more_epochs